In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv


In [2]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

**Profiling Report by ydata_profiling** (Included along with the Notebook)

In [3]:
'''
from ydata_profiling import ProfileReport

cols = ['ID','Source', 'Severity', 'Start_Time', 'End_Time', 'Start_Lat',
       'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)',
       'Street', 'City', 'County', 'State', 'Zipcode', 'Timezone',
       'Airport_Code', 'Weather_Timestamp', 'Temperature(F)', 'Wind_Chill(F)',
       'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction',
       'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity',
       'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway',
       'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal',
       'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight',
       'Astronomical_Twilight']

df = pd.read_csv('/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv',
                index_col='ID',
                usecols = cols,
                dtype = {
                    'Severity' : np.int8
                },
                parse_dates=['Start_Time','End_Time','Weather_Timestamp']
                )

profile = ProfileReport(df,title='US Accident Data Profiling')
profile.to_file('US Accident Data Profiling.html')
'''

"\nfrom ydata_profiling import ProfileReport\n\ncols = ['ID','Source', 'Severity', 'Start_Time', 'End_Time', 'Start_Lat',\n       'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)',\n       'Street', 'City', 'County', 'State', 'Zipcode', 'Timezone',\n       'Airport_Code', 'Weather_Timestamp', 'Temperature(F)', 'Wind_Chill(F)',\n       'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction',\n       'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity',\n       'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway',\n       'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal',\n       'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight',\n       'Astronomical_Twilight']\n\ndf = pd.read_csv('/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv',\n                index_col='ID',\n                usecols = cols,\n                dtype = {\n                    'Severity' : np.int8\n                },\n            

In [4]:
'''
cols = ['ID','Source', 'Severity', 'Start_Time', 'End_Time', 'Start_Lat',
       'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)',
       'Street', 'City', 'County', 'State', 'Zipcode', 'Timezone',
       'Airport_Code', 'Weather_Timestamp', 'Temperature(F)', 'Wind_Chill(F)',
       'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction',
       'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity',
       'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway',
       'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal',
       'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight',
       'Astronomical_Twilight']

Profling Reveals-

1) Zipcode columns has too many categories,
   plus all information conveyed by it is already in the tuple (Street,City,County,State)

2) (Start_Lat,Start_Lng,End_Lat,End_Lng) is road information essentially, but End infomation is 44% missing at times.
   So Based on correlation between Start & End tuples, we can only use Start to identify road.

3) Distance in Miles, requires a specialized binning scheme due to high frequency of 0

4) Airport_Code tells the nearest weather station, but that data is not usefull as no reliability estimate is given;
   so all weather realated data is assumed related data is assumed relaible and this col is not needed.

5) We will ignore Weather_Timestamp on the basis of the above asumption.

6) Weather_Condition is highly correlated with the tuple
   (Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),precipitation(in))
   So we will decide later which to use, depending on the algorithm used.

7) Binary columns, (Amenity,Bump,Crosing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal)
   give information about road and possible reasons for acidents.
   Note: Turning_Loop was dropped as it has 0% values for True

8) The columns, (Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight) are highly correlated and will be encoded into a singal columns,
   or kept same depending on algorithm.
'''

"\ncols = ['ID','Source', 'Severity', 'Start_Time', 'End_Time', 'Start_Lat',\n       'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)',\n       'Street', 'City', 'County', 'State', 'Zipcode', 'Timezone',\n       'Airport_Code', 'Weather_Timestamp', 'Temperature(F)', 'Wind_Chill(F)',\n       'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction',\n       'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity',\n       'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway',\n       'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal',\n       'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight',\n       'Astronomical_Twilight']\n\nProfling Reveals-\n\n1) Zipcode columns has too many categories,\n   plus all information conveyed by it is already in the tuple (Street,City,County,State)\n\n2) (Start_Lat,Start_Lng,End_Lat,End_Lng) is road information essentially, but End infomation is 44% missing at times.\n   So Based on correlation bet

In [5]:
'''
Features Schema - 

Source = Label Encoded (1..3) // -1 -> null, assert handles that

Severity = Ordinal Encoded (1..4)

(Start_Time,End_Time) = (Start_Date, Start_Hour, End_Hour)
> Start_Date = DateTime
> Start_Hour = Cyclically Encoded
> End_Hour = Cyclically Encoded

(Start_Lat,Start_Lng,End_Lat,End_Lng) = Categorical, Logically Grouped

Distance(mi) = Possibly Outliers, 5 number summary was very skewed. No Encoding done for now

(Street,City,County,State) = Categorical, Logically Grouped

Timezone = Categorical (E,P,C,M), Label Encoding Specifically not Prefered

(Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Precipitation(in)) = Numerical(Float64), Logically Grouped -|
(Wind_Direction,Weather_Condition) = Categorical, Logically Grouped                                                                              -|
                                                                                                                                  Logically Grouped

(Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal) = Boolean, Logically Grouped

(Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight) = Boolean, Logically Grouped
'''

'\nFeatures Schema - \n\nSource = Label Encoded (1..3) // -1 -> null, assert handles that\n\nSeverity = Ordinal Encoded (1..4)\n\n(Start_Time,End_Time) = (Start_Date, Start_Hour, End_Hour)\n> Start_Date = DateTime\n> Start_Hour = Cyclically Encoded\n> End_Hour = Cyclically Encoded\n\n(Start_Lat,Start_Lng,End_Lat,End_Lng) = Categorical, Logically Grouped\n\nDistance(mi) = Possibly Outliers, 5 number summary was very skewed. No Encoding done for now\n\n(Street,City,County,State) = Categorical, Logically Grouped\n\nTimezone = Categorical (E,P,C,M), Label Encoding Specifically not Prefered\n\n(Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Precipitation(in)) = Numerical(Float64), Logically Grouped -|\n(Wind_Direction,Weather_Condition) = Categorical, Logically Grouped                                                                              -|\n                                                                                                          

In [6]:
plcols = ['Source', 'Severity', 'Start_Time', 'End_Time', 'Start_Lat',
       'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)',
       'Street', 'City', 'County', 'State', 'Timezone',
       'Temperature(F)', 'Wind_Chill(F)',
       'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction',
       'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity',
       'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway',
       'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal',
       'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight',
       'Astronomical_Twilight']

# ID, country, description, turning_loop, Zipcode were dropped

In [7]:
lf = (
    pl.scan_csv('/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv',
                schema_overrides = {
                    'Severity' : pl.UInt8
                },
                try_parse_dates = True)
    .select(plcols)
)

In [8]:
'''Initial Schema'''

schema = lf.collect_schema()
print("Columns:", len(schema))
print()

for name, dtype in schema.items():
    print(f"{name:22} {dtype}")

Columns: 39

Source                 String
Severity               UInt8
Start_Time             Datetime(time_unit='us', time_zone=None)
End_Time               Datetime(time_unit='us', time_zone=None)
Start_Lat              Float64
Start_Lng              Float64
End_Lat                String
End_Lng                String
Distance(mi)           Float64
Street                 String
City                   String
County                 String
State                  String
Timezone               String
Temperature(F)         Float64
Wind_Chill(F)          Float64
Humidity(%)            Float64
Pressure(in)           Float64
Visibility(mi)         Float64
Wind_Direction         String
Wind_Speed(mph)        Float64
Precipitation(in)      Float64
Weather_Condition      String
Amenity                Boolean
Bump                   Boolean
Crossing               Boolean
Give_Way               Boolean
Junction               Boolean
No_Exit                Boolean
Railway                Boolean
Rou

**Feature Enginnering**

In [9]:
'''Label Encoding the Source column'''

lf = lf.with_columns(
    pl.col('Source')
    .replace({
        'Source1' : 1,
        'Source2' : 2,
        'Source3' : 3
    })
    .fill_null(-1)
    .cast(pl.UInt8)
    .alias('Source')
)

# Since we are asuming all accidents have a Source, we are hard coding this behaviour
# This assumption implies the existance of the rest of the data, so only tested here
assert not (
    lf.select((pl.col('Source') == -1).any())
    .collect()
    .item()
), 'Source is Null in a row'

In [10]:
'''Feature Enginnering (Start_Time, End_Time)
-> (Start_Date, Start_Hour, End_Hour) this will help feature engineer for Planned Models'''



'Feature Enginnering (Start_Time, End_Time)\n-> (Start_Date, Start_Hour, End_Hour) this will help feature engineer for Planned Models'

In [11]:
'''Resolving Nulls in End_Lat, End_Lng'''

'''
unique_vals = (
    lf
    .select(pl.col("End_Lat").unique())
    .collect()
    .to_series()
    .to_list()
)
assert not (None in unique_vals), 'None is in End_Lat'
'''
'''
unique_vals = (
    lf
    .select(pl.col("End_Lng").unique())
    .collect()
    .to_series()
    .to_list()
)
assert not (None in unique_vals), 'None is in End_Lng'
'''
# Both have Nulls, lets resolve them

lf = lf.with_columns(
    pl.col('End_Lat')
    .cast(pl.Float64),
    pl.col('End_Lng')
    .cast(pl.Float64)
)

In [12]:
'''Categorical Conversion of (Street,City,County,State), Encoding will not be changed for now'''
lf = lf.with_columns(
    pl.col('Street')
    .cast(pl.Categorical),
    pl.col('City')
    .cast(pl.Categorical),
    pl.col('County')
    .cast(pl.Categorical),
    pl.col('State')
    .cast(pl.Categorical)
)

In [13]:
'''Label Encoding Timezone'''
lf = lf.with_columns(
    pl.col('Timezone')
    .replace({
        'US/Eastern' : 'E',
        'US/Pacific' : 'P',
        'US/Central' : 'C',
        'US/Mountain' : 'M'
    })
    .cast(pl.Categorical)
)

In [14]:
lf.select(pl.len()).collect().item() # Current Size

7728394

In [15]:
'''Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition'''

weather_data = ['Temperature(F)', 'Wind_Chill(F)','Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 
                'Wind_Direction','Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition']

'''
for _ in weather_data:
    assert not (
        lf.select(pl.col(_).is_null().any())
        .collect()
        .item()
    )
'''

lf = lf.drop_nulls(subset=weather_data)

In [16]:
lf.select(pl.len()).collect().item() # Now How many items remain
'''67.77% Data Remain'''

'67.77% Data Remain'

In [17]:
'''Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Precipitation(in)'''

# I leave the scaling to each model when need be

'Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Precipitation(in)'

In [18]:
'''Wind_Direction'''
unique_vals = (
    lf.select(pl.col('Wind_Direction').unique())
    .collect()
    .to_series()
    .to_list()
)

len(unique_vals)


lf = lf.with_columns(
    pl.col('Wind_Direction')
    .cast(pl.Categorical)
)
# This is rich data, we will leave it as it is

In [19]:
'''Weather_Condition'''

unique_vals = (
    lf.select(pl.col('Weather_Condition').unique())
    .collect()
    .to_series()
    .to_list()
)

len(unique_vals)

lf = lf.with_columns(
    pl.col('Weather_Condition')
    .cast(pl.Categorical)
)

# This is rich data, we will leave it as it is

In [20]:
'''These are already Boolean'''

binary_cols = ['Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit',
               'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal']

In [21]:
'''We will encode them as booleans'''

lf = lf.with_columns([
    pl.col(c).replace({
        'Night' : 0,
        'Day' : 1
    })
    .cast(pl.Boolean)
    for c in ['Sunrise_Sunset','Civil_Twilight','Nautical_Twilight','Astronomical_Twilight']
])

In [22]:
schema = lf.collect_schema()
print("Columns:", len(schema))
print()

for name, dtype in schema.items():
    print(f"{name:22} {dtype}")

Columns: 39

Source                 UInt8
Severity               UInt8
Start_Time             Datetime(time_unit='us', time_zone=None)
End_Time               Datetime(time_unit='us', time_zone=None)
Start_Lat              Float64
Start_Lng              Float64
End_Lat                Float64
End_Lng                Float64
Distance(mi)           Float64
Street                 Categorical
City                   Categorical
County                 Categorical
State                  Categorical
Timezone               Categorical
Temperature(F)         Float64
Wind_Chill(F)          Float64
Humidity(%)            Float64
Pressure(in)           Float64
Visibility(mi)         Float64
Wind_Direction         Categorical
Wind_Speed(mph)        Float64
Precipitation(in)      Float64
Weather_Condition      Categorical
Amenity                Boolean
Bump                   Boolean
Crossing               Boolean
Give_Way               Boolean
Junction               Boolean
No_Exit                Boolea

In [23]:
# from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, FunctionTransformer, PowerTransformer
# from sklearn.compose import ColumnTransformer

In [24]:
# sns.countplot(data = df[['Distance(mi)']],x='Distance(mi)')